In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, r2_score

DATA_PATH = "/Users/sanjana/Desktop/Hype-Predictor/Data"

df = pd.read_csv(f"{DATA_PATH}/enhanced_data.csv")

df.head()

,product,trends_score,reddit_score,youtube_score,news_score,momentum_score,trend_label
0,Air Jordan 11,33.114504,44,2854358,96,0.000000,📉 Losing Hype
1,Nvidia RTX 5090,9.385496,4812,15542048,100,29.166667,📉 Losing Hype
2,Owala FreeSip,11.656489,3780,1722224,27,100.000000,🔥 Trending Up
3,PS5 Pro,10.225191,41159,3643056,98,12.500000,📉 Losing Hype
4,iPhone 17,4.362595,12814,8674851,99,19.444444,📉 Losing Hype


In [4]:
#Recompute Hype Score (same as before — keep consistency)
cols = ["trends_score", "reddit_score", "youtube_score", "news_score"]

# Z-score
df_z = df.copy()
for col in cols:
    df_z[col] = (df[col] - df[col].mean()) / df[col].std()

# Scale
df_scaled = df_z.copy()
for col in cols:
    df_scaled[col] = 100 * (df_z[col] - df_z[col].min()) / (df_z[col].max() - df_z[col].min())

# Weights
weights = {
    "trends_score": 0.25,
    "reddit_score": 0.20,
    "youtube_score": 0.30,
    "news_score": 0.25
}

df_scaled["hype_score"] = (
    df_scaled["trends_score"] * weights["trends_score"] +
    df_scaled["reddit_score"] * weights["reddit_score"] +
    df_scaled["youtube_score"] * weights["youtube_score"] +
    df_scaled["news_score"] * weights["news_score"]
)

# Consistency
df_scaled["consistency_std"] = df_scaled[cols].std(axis=1)

df_scaled["consistency_score"] = 100 * (
    1 - (df_scaled["consistency_std"] - df_scaled["consistency_std"].min()) /
    (df_scaled["consistency_std"].max() - df_scaled["consistency_std"].min())
)

df_scaled["hype_score_adjusted"] = (
    df_scaled["hype_score"] * 0.90 +
    df_scaled["consistency_score"] * 0.10
)

df = df_scaled

In [6]:
#Create Prediction Target
df = df.sort_values("hype_score_adjusted", ascending=False).reset_index(drop=True)

df["future_hype"] = df["hype_score_adjusted"].shift(-1)
df = df.dropna()

In [8]:
#Features (THIS is where all your data comes together)
features = [
    "trends_score",
    "reddit_score",
    "youtube_score",
    "news_score",
    "momentum_score"
]

X = df[features]
y = df["future_hype"]

In [14]:
model = RandomForestRegressor(random_state=42)
model.fit(X, y)

preds = model.predict(X)

from sklearn.metrics import mean_absolute_error

print("MAE:", mean_absolute_error(y, preds))

MAE: 4.832951512437204


In [16]:
importance = pd.Series(model.feature_importances_, index=features)
importance = importance.sort_values(ascending=False)

print(importance)

trends_score      0.606943
reddit_score      0.144081
momentum_score    0.103484
news_score        0.080814
youtube_score     0.064678
dtype: float64


In [18]:
# 📊 Feature Importance Insights#
## The model reveals that Google Trends is the most significant predictor of product hype, contributing over 60% of the importance. This suggests that search behavior is a strong leading indicator of consumer interest.
## Reddit activity is the second most important factor, indicating that community discussions play a key role in sustaining hype.
## Momentum, derived from time-series trends data, also contributes meaningfully, showing that growth rate is an important signal for emerging products.
## News and YouTube signals have lower importance in this dataset, suggesting they may act as secondary or lagging indicators rather than primary drivers of hype.
Due to the small dataset size, these results should be interpreted as directional insights rather than definitive conclusions.

SyntaxError: invalid syntax (3490974547.py, line 6)